In [10]:
import torch

rankobject = {1: 123144}
for key in rankobject.keys():
    print(rankobject[key])

123144


In [ ]:
import torch
import torch.distributed as dist
import triton
import triton.language as tl

@triton.jit
def add_triton_inner(a_ptr, b_ptr, output_ptr, a_stride_0, a_stride_1, a_stride_2, b_stride_0, b_stride_1, b_stride_2,
                         X_DIM, Y_DIM, BLOCK_X: tl.constexpr, BLOCK_Y: tl.constexpr):
  y_id = tl.program_id(0)
  x_id = tl.program_id(1)
  
  col_x = x_id*BLOCK_X
  row_y = y_id*BLOCK_Y
  offset_y = row_y + tl.arange(0, BLOCK_Y)
  offset_x = col_x + tl.arange(0, BLOCK_X)
  offset = offset_y[:, None]*a_stride_1 + offset_x[None,:]*a_stride_2
  a = tl.load(a_ptr+offset);
  b = tl.load(b_ptr+offset);
  c = a + b
  tl.store(output_ptr+offset, c)

def add_triton_dist(a, b, output, BATCH_SIZE, X_DIM, Y_DIM, BLOCK_X, BLOCK_Y):
  assert a.shape == b.shape, "Addition Impossible"
  a_stride_0 = a.stride(0)
  a_stride_1 = a.stride(1)
  a_stride_2 = a.stride(2)
  b_stride_0 = b.stride(0)
  b_stride_1 = b.stride(1)
  b_stride_2 = b.stride(2)

  y_axis = Y_DIM * BATCH_SIZE
  x_axis = X_DIM
  grid_0 = y_axis // BLOCK_Y
  grid_1 = X_DIM // BLOCK_X
  grid = (grid_0, grid_1)
  add_triton_inner[grid](a, b, output, a_stride_0, a_stride_1, a_stride_2, b_stride_0, b_stride_1, b_stride_2, X_DIM, Y_DIM, BLOCK_X, BLOCK_Y)

DEVICE = triton.runtime.driver.active.get_active_torch_device()
# DEVICE = torch.device("cuda:0")
BATCH_SIZE = 10
X_DIM = 32
Y_DIM = 32
a = torch.rand(BATCH_SIZE, X_DIM, Y_DIM, device=DEVICE, dtype=torch.float32)
b = torch.rand(BATCH_SIZE, X_DIM, Y_DIM, device=DEVICE, dtype=torch.float32)
output = torch.zeros(BATCH_SIZE, X_DIM, Y_DIM, device=DEVICE, dtype=torch.float32)
BLOCK_X = 4
BLOCK_Y = 4

# Single triton kernel
# add_triton_dist(a, b, output, BATCH_SIZE, X_DIM, Y_DIM, BLOCK_X, BLOCK_Y)
# Multi-triton kernel
